In [ ]:
import numpy as np
import pandas as pd


In [2]:
movies=pd.read_csv("tmdb_5000_movies.csv")
credits=pd.read_csv("tmdb_5000_credits.csv")


In [ ]:
movies.head()

In [ ]:
credits.head(1)

In [ ]:
credits.head(1)['cast'].values

In [ ]:
credits.head(1)['crew'].values

In [7]:
movies.merge(credits,on='title').shape

(4809, 23)

In [8]:
credits.shape


(4803, 4)

In [9]:
movies.shape

(4803, 20)

In [10]:
movies=movies.merge(credits,on='title')

In [ ]:
movies.head()

In [12]:
#generes we should keep 
#id also
#keyword
#title
#overview
#cast
#crew


In [ ]:
movies['original_language'].value_counts()

In [ ]:
movies.info()

In [15]:
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies.head()

In [19]:
movies.isnull().sum()

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [18]:
movies.dropna(inplace=True)

In [20]:
movies.duplicated().sum()

0

In [21]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [24]:
import ast
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
        return L

In [25]:
movies['genres']=movies['genres'].apply(convert)

In [ ]:
movies.head()

In [27]:
movies['keywords']=movies['keywords'].apply(convert)

In [ ]:
movies.head()

In [ ]:
movies['cast'][0]

In [32]:
import ast

def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L


In [ ]:
movies['cast'].apply(convert3)

In [34]:
movies['cast']=movies['cast'].apply(convert3)

In [ ]:
movies.head()

In [ ]:
movies['crew'][0]

In [37]:
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L        
            

In [39]:
movies['crew']=movies['crew'].apply(fetch_director)

In [ ]:
movies.head()

In [ ]:
movies['overview'][0]

In [43]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [ ]:
movies.head()

In [48]:
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x] if isinstance(x, list) else [])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x] if isinstance(x, list) else [])
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x] if isinstance(x, list) else [])
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x] if isinstance(x, list) else [])

In [ ]:
movies.head()

In [50]:
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [ ]:
movies.head()

In [53]:
new_df=movies[['movie_id','title','tags']]

In [ ]:
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df.head()

In [58]:
new_df['tags'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action cultureclash SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [ ]:
new_df['tags']=new_df['tags'].apply(lambda x:x.lower())

In [ ]:
new_df.head()

In [61]:
new_df['tags'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action cultureclash samworthington zoesaldana sigourneyweaver jamescameron'

In [ ]:
new_df['tags'][1]

In [63]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')

In [64]:
vectors=cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors

In [86]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [87]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [ ]:
!pip install nltk

In [70]:
import nltk

In [71]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [79]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return" ".join(y)    

In [ ]:
new_df['tags']=new_df['tags'].apply(stem)

In [83]:
['loved','loving','love']
['love','love','love']

['love', 'love', 'love']

In [84]:
ps.stem('dancing')

'danc'

In [85]:
stem('in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action cultureclash samworthington zoesaldana sigourneyweaver jamescameron')

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action cultureclash samworthington zoesaldana sigourneyweav jamescameron'

In [90]:
from sklearn.metrics.pairwise import cosine_similarity

In [94]:
similarity=cosine_similarity(vectors)

In [ ]:
similarity

In [ ]:
sorted(similarity[1] ,reverse=True)  #for descending order

In [ ]:
list(enumerate(similarity[0]))  #index and the distance

In [110]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

[(3608, 0.2525907427704613),
 (4405, 0.25),
 (529, 0.19062321291575585),
 (311, 0.17407765595569785),
 (151, 0.16666666666666666)]

In [125]:
def recommend(movie):
    movie_index=new_df[new_df['title']=='Batman Begins'].index[0]
    #distances=similarity[movie_index]
    movies_list=sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])

    for i in distances[1:6]:
        print(new_df.iloc[i[0]].title)
    

In [117]:
recommend('Batman Begins')

Apollo 18
The Helix... Loaded
Tears of the Sun
The Adventures of Pluto Nash
Beowulf


In [116]:
new_df.iloc[1216].title

'Aliens vs Predator: Requiem'

In [118]:
import pickle

In [119]:
pickle.dump(new_df,open('movies.pkl','wb'))

In [120]:
new_df['title'].values

array(['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre',
       ..., 'Signed, Sealed, Delivered', 'Shanghai Calling',
       'My Date with Drew'], dtype=object)

In [121]:
pickle.dump(new_df.to_dict(),open('movie_dict.pkl','wb'))

In [122]:
pickle.dump(similarity,open('similarity.pkl','wb'))

In [123]:
recommend('Gandhi')

Apollo 18
The Helix... Loaded
Tears of the Sun
The Adventures of Pluto Nash
Beowulf
